In [4]:
import pandas as pd
from sqlalchemy import create_engine

In [5]:
engine = create_engine('mysql+mysqlconnector://root@localhost:3306/project')

In [6]:
def run_sql_query(query):
    return pd.read_sql(query, engine)


### Category Revenue
 Which product categories generate the highest total revenue?

In [8]:
run_sql_query(
"""
SELECT 
    category, 
    SUM(purchase_amount) AS total_revenue
FROM clean_customer_data
GROUP BY category
ORDER BY total_revenue DESC;
"""
)

,category,total_revenue
0,Clothing,104320.0
1,Accessories,74200.0
2,Footwear,36093.0
3,Outerwear,18524.0


### Key Insights:

- Clothing is the highest revenue-generating category, contributing significantly more than other categories.
- Accessories represent the second-largest revenue segment.
- Footwear and Outerwear contribute comparatively lower revenue.

This indicates that Clothing is the core business driver and should remain the primary focus for inventory planning, marketing campaigns, and customer engagement strategies.


### Average Purchase Value
What is the average purchase amount per category?

In [9]:
run_sql_query(
"""
SELECT 
    category, 
    AVG(purchase_amount) AS avg_purchase_amount
FROM clean_customer_data
GROUP BY category
ORDER BY avg_purchase_amount DESC;
"""
 )

,category,avg_purchase_amount
0,Footwear,60.2554
1,Clothing,60.0576
2,Accessories,59.8387
3,Outerwear,57.1728


### Key Insights:

- Footwear has the highest average purchase value per transaction.
- Clothing and Accessories have very similar average spending.
- Outerwear has the lowest average transaction value.

Although Clothing generates the highest total revenue (from previous analysis),
Footwear customers tend to spend slightly more per purchase.


### Customer Count by Gender

How many customers belong to each gender?


In [10]:
run_sql_query(
"""
SELECT 
    gender, 
    COUNT(DISTINCT customer_id) AS customer_count
FROM clean_customer_data
GROUP BY gender;
"""
)

,gender,customer_count
0,Female,1248
1,Male,2652


### Key Insights:

- Male customers represent nearly 68% of total customers.
- Female customers represent around 32%.
- The business has a significantly higher male customer base.


### Seasonal Revenue

What is the total purchase amount for each season?

In [11]:
run_sql_query(
"""
SELECT 
    season, 
    SUM(purchase_amount) AS total_revenue
FROM clean_customer_data
GROUP BY season
ORDER BY total_revenue DESC;
"""
)

,season,total_revenue
0,Fall,60018.0
1,Spring,58679.0
2,Winter,58607.0
3,Summer,55833.0


### Key Insights:

- Fall generates the highest revenue (60,018).
- Summer generates the lowest revenue (55,833).
- Revenue variation across seasons is relatively small.


### Payment Method Usage

How many transactions were made using each payment method?

In [31]:
run_sql_query(
"""
SELECT 
    payment_method,
    COUNT(*) AS transaction_count,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(),
        2
    ) AS usage_percentage
FROM clean_customer_data
GROUP BY payment_method
ORDER BY transaction_count DESC;

"""
)

,payment_method,transaction_count,usage_percentage
0,PayPal,677,17.36
1,Credit Card,671,17.21
2,Cash,670,17.18
3,Debit Card,636,16.31
4,Venmo,634,16.26
5,Bank Transfer,612,15.69


### Key Insights:

- PayPal is the most used payment method (17.36%).
- Payment usage is very evenly distributed.
- No single dominant payment method.


### Top Category per Season

Which category generates the highest revenue in each season?


In [13]:
run_sql_query(
"""
WITH season_category_revenue AS (
    SELECT 
        season,
        category,
        SUM(purchase_amount) AS total_revenue,
        RANK() OVER (
            PARTITION BY season 
            ORDER BY SUM(purchase_amount) DESC
        ) AS revenue_rank
    FROM clean_customer_data
    GROUP BY season, category
)
SELECT
    season,
    category,
    total_revenue
FROM season_category_revenue
WHERE revenue_rank = 1 ORDER BY total_revenue DESC;
"""
)

,season,category,total_revenue
0,Spring,Clothing,27692.0
1,Winter,Clothing,27274.0
2,Fall,Clothing,26220.0
3,Summer,Clothing,23134.0


### Key Insights:

- Clothing is the highest revenue category in ALL seasons.
- Seasonal variation exists, but dominance remains consistent.
- Spring generates the highest Clothing revenue.


### Discount Impact

Compare the average purchase amount for transactions:

with discount

without discount

In [32]:
run_sql_query(
"""
SELECT 
    discount_applied,
    COUNT(*) AS total_orders,
    SUM(purchase_amount) AS total_revenue,
    AVG(purchase_amount) AS avg_order_value
FROM clean_customer_data
GROUP BY discount_applied;
"""
)

,discount_applied,total_orders,total_revenue,avg_order_value
0,No,2223,133670.0,60.1305
1,Yes,1677,99467.0,59.3125


### Promo Code Effectiveness

Does using a promo code increase the average purchase amount?


In [15]:
run_sql_query(
"""
SELECT 
    promo_code_used, 
    AVG(purchase_amount) AS avg_purchase_amount
FROM clean_customer_data
GROUP BY promo_code_used;
"""
)

,promo_code_used,avg_purchase_amount
0,No,60.1305
1,Yes,59.3125


### Age Group Spending

Which age group contributes the most total revenue?

In [16]:
run_sql_query(
"""
SELECT 
    age_group, 
    SUM(purchase_amount) AS total_revenue
FROM clean_customer_data
GROUP BY age_group
ORDER BY total_revenue DESC;
"""
)

,age_group,total_revenue
0,Middle Aged,89445.0
1,Young,65842.0
2,Senior,43220.0
3,Young Adult,34630.0


### Location-wise Revenue

List the top 5 locations by total purchase amount.

In [17]:
run_sql_query(
"""
select 
    location, 
    sum(purchase_amount) as total_revenue 
    from clean_customer_data 
    group by location 
    order by total_revenue desc 
    limit 5
"""
)

,location,total_revenue
0,Montana,5784.0
1,Illinois,5617.0
2,California,5605.0
3,Idaho,5587.0
4,Nevada,5514.0


### Top 5 Revenue-Generating Locations

This analysis highlights the top five locations contributing the highest
total purchase revenue, supporting region-based sales and marketing strategies.


In [18]:
run_sql_query(
"""
WITH top_ranked_items AS (
    SELECT
        category,
        item_purchased,
        SUM(purchase_amount) AS total_revenue,
        RANK() OVER (
            PARTITION BY category 
            ORDER BY SUM(purchase_amount) DESC
        ) AS rank
    FROM clean_customer_data
    GROUP BY category, item_purchased
)
SELECT
    category,
    item_purchased,
    total_revenue
FROM top_ranked_items
WHERE rank = 1;
"""
)

,category,item_purchased,total_revenue
0,Accessories,Jewelry,10010.0
1,Clothing,Blouse,10410.0
2,Footwear,Shoes,9240.0
3,Outerwear,Coat,9275.0


 ### Most Popular Product

For each category, find the most frequently purchased item.


In [19]:
run_sql_query(
"""
with top_sold_items as (
    select
        category,
        item_purchased,
        COUNT(*) as purchase_count,
        RANK() OVER(PARTITION BY category ORDER BY COUNT(*) desc) as item_rank
        from clean_customer_data
        group by category, item_purchased
)
select
    category,
    item_purchased,
    purchase_count
    from top_sold_items
    where item_rank = 1
"""
)

,category,item_purchased,purchase_count
0,Accessories,Jewelry,171
1,Clothing,Blouse,171
2,Clothing,Pants,171
3,Footwear,Sandals,160
4,Outerwear,Jacket,163


### Revenue by Purchase Frequency

How much total revenue is generated by each frequency_of_purchases group?


In [20]:
run_sql_query(
"""
SELECT 
    frequency_of_purchases, 
    SUM(purchase_amount) AS total_revenue
FROM clean_customer_data
GROUP BY frequency_of_purchases
ORDER BY total_revenue DESC;
"""
)

,frequency_of_purchases,total_revenue
0,Every 3 Months,35088.0
1,Annually,34419.0
2,Quarterly,33771.0
3,Bi-Weekly,33256.0
4,Monthly,32810.0
5,Fortnightly,32007.0
6,Weekly,31786.0


### Category Preference by Gender

For each gender, which category contributes the highest revenue?

In [21]:
run_sql_query(
"""
WITH top_revenue_by_gender AS (
    SELECT 
        gender, 
        category, 
        SUM(purchase_amount) AS total_revenue,
        RANK() OVER (
            PARTITION BY gender 
            ORDER BY SUM(purchase_amount) DESC
        ) AS gender_rank
    FROM clean_customer_data
    GROUP BY category, gender
)
SELECT
    category,
    gender,
    total_revenue
FROM top_revenue_by_gender
WHERE gender_rank = 1;
"""
)

,category,gender,total_revenue
0,Clothing,Female,33636.0
1,Clothing,Male,70684.0


### Rank Customers by Spending

Rank customers based on total purchase amount (highest first).

In [22]:
run_sql_query(
"""
WITH customer_total_spend AS (
    SELECT 
        customer_id, 
        SUM(purchase_amount) AS total_revenue
    FROM clean_customer_data
    GROUP BY customer_id   
)
SELECT 
    customer_id, 
    total_revenue,
    RANK() OVER (ORDER BY total_revenue DESC) AS customer_rank
FROM customer_total_spend;
"""
)

,customer_id,total_revenue,customer_rank
0,20,146.0,1
1,3838,100.0,2
2,862,100.0,2
3,1406,100.0,2
4,43,100.0,2
...,...,...,...
3895,1066,20.0,3849
3896,3530,20.0,3849
3897,3639,20.0,3849
3898,868,20.0,3849


### Top 3 Products per Category

Find the top 3 products in each category by revenue.


In [24]:
run_sql_query(
"""
WITH top3_items_by_category AS (
    SELECT 
        category,
        item_purchased,
        SUM(purchase_amount) AS total_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY category 
            ORDER BY SUM(purchase_amount) DESC
        ) AS items_rank 
    FROM clean_customer_data
    GROUP BY category, item_purchased
) 
SELECT
    category,
    item_purchased,
    total_revenue
FROM top3_items_by_category
WHERE items_rank <= 3
ORDER BY category, items_rank;
"""
)

,category,item_purchased,total_revenue
0,Accessories,Jewelry,10010.0
1,Accessories,Sunglasses,9649.0
2,Accessories,Belt,9635.0
3,Clothing,Blouse,10410.0
4,Clothing,Shirt,10332.0
5,Clothing,Dress,10320.0
6,Footwear,Shoes,9240.0
7,Footwear,Sandals,9200.0
8,Footwear,Boots,9018.0
9,Outerwear,Coat,9275.0


### Best Category per Age Group
For each age group, find the category with highest revenue.

In [25]:
run_sql_query(
"""
WITH top_revenue_by_age_group AS (
    SELECT 
        category, 
        age_group, 
        SUM(purchase_amount) AS total_revenue,
        RANK() OVER (
            PARTITION BY age_group 
            ORDER BY SUM(purchase_amount) DESC
        ) AS age_group_rank
    FROM clean_customer_data
    GROUP BY category, age_group
)
SELECT 
    category,
    age_group,
    total_revenue
FROM top_revenue_by_age_group
WHERE age_group_rank = 1;
"""
)

,category,age_group,total_revenue
0,Clothing,Middle Aged,39758.0
1,Clothing,Senior,19477.0
2,Clothing,Young,28864.0
3,Clothing,Young Adult,16221.0


### High Spenders with Discounts

Which customers used a discount but still spent more than the overall average purchase amount?

In [26]:
run_sql_query(
"""
SELECT 
    customer_id,
    SUM(purchase_amount) AS total_revenue
FROM clean_customer_data
WHERE discount_applied = 'yes'
GROUP BY customer_id
HAVING SUM(purchase_amount) > (
    SELECT AVG(purchase_amount)
    FROM clean_customer_data
);

"""
)

,customer_id,total_revenue
0,2,64.0
1,3,73.0
2,4,90.0
3,7,85.0
4,9,97.0
...,...,...
834,1667,64.0
835,1671,73.0
836,1673,73.0
837,1674,62.0


### Top Rated Products

Which are the top 5 products with the highest average review rating?

In [27]:
run_sql_query( 
"""
SELECT 
    item_purchased, 
    ROUND(AVG(review_rating), 2) AS avg_rating
FROM clean_customer_data
GROUP BY item_purchased
ORDER BY avg_rating DESC
LIMIT 5;
"""
)

,item_purchased,avg_rating
0,Gloves,3.86
1,Sandals,3.84
2,Boots,3.82
3,Hat,3.80
4,Handbag,3.78


### Shipping Type Comparison

Compare the average purchase amount between:

Standard Shipping

Express Shipping

In [28]:
run_sql_query(
"""
SELECT 
    shipping_type, 
    ROUND(AVG(purchase_amount), 2) AS avg_purchase_amount
FROM clean_customer_data
WHERE shipping_type IN('Express', 'Standard')
GROUP BY shipping_type;
"""
)

,shipping_type,avg_purchase_amount
0,Express,60.48
1,Standard,58.55


### Do subscribed customers spend more than non-subscribers?
Compare:

Average purchase amount

Total revenue

In [29]:
run_sql_query(
"""
SELECT 
    subscription_status, 
    SUM(purchase_amount) AS total_revenue, 
    AVG(purchase_amount) AS avg_revenue
FROM clean_customer_data 
GROUP BY subscription_status;
"""
)

,subscription_status,total_revenue,avg_revenue
0,No,170436.0,59.8651
1,Yes,62701.0,59.5451


### Discount Dependency

Which 5 products have the highest percentage of purchases where discounts were applied?

In [30]:
run_sql_query( 
"""
SELECT
    item_purchased,
    ROUND(
        SUM(CASE WHEN discount_applied = 'yes' THEN purchase_amount ELSE 0 END)
        / SUM(purchase_amount) * 100,
        2
    ) AS discount_percentage
FROM clean_customer_data
GROUP BY item_purchased
ORDER BY discount_percentage DESC
LIMIT 5;
"""
)

,item_purchased,discount_percentage
0,Sneakers,50.05
1,Hat,49.02
2,Sweater,48.72
3,Pants,48.23
4,Coat,48.19


### Customer Segmentation  
Classified customers into New, Returning , and Loyal segments based on  purchase history.

In [9]:
run_sql_query( 
"""
SELECT 
	customer_segmentation, 
    COUNT(customer_id) as number_of_customer 
    FROM clean_customer_data
    GROUP BY customer_segmentation

"""
)

,customer_segmentation,number_of_customer
0,Loyal,3116
1,New,83
2,Returning,701
